# One-time frozen predictive held-out test

Run all cells in a fresh Colab runtime. The notebook first freezes and displays the decision and all 20 validation-selected checkpoints without opening test data. Test inference starts only after the exact confirmation phrase is entered. No training or checkpoint selection occurs here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = Path('/content/VitalDB-Feature-Selection')
DRIVE_PROJECT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')

if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', 'main', 'origin/main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
print(subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
import torch

DATASET_DIR = DRIVE_PROJECT / 'data/modeling/full'
ANALYSIS_MANIFEST = DRIVE_PROJECT / 'outputs/frozen_candidate_retraining_validation_only/analysis/analysis_manifest.json'
STRICT_ROOT = DRIVE_PROJECT / 'outputs/frozen_candidate_retraining_validation_only/strict_consensus'
FULL17_ROOT = DRIVE_PROJECT / 'outputs/group_retraining_validation_only/full17'
DECISION_TEMPLATE_DIR = REPO_DIR / 'outputs/frozen_predictive_decision_30s'
DECISION_DIR = DRIVE_PROJECT / 'outputs/frozen_predictive_decision_30s'
OUTPUT_DIR = DRIVE_PROJECT / 'outputs/frozen_predictive_test_evaluation_30s'
required = [DATASET_DIR, ANALYSIS_MANIFEST, STRICT_ROOT, FULL17_ROOT, DECISION_TEMPLATE_DIR]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required Drive/repository inputs: {missing}')
print({'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'})

In [ ]:
COMMON_ARGS = [
    '--decision-template-dir', str(DECISION_TEMPLATE_DIR),
    '--decision-dir', str(DECISION_DIR),
    '--analysis-manifest', str(ANALYSIS_MANIFEST),
    '--dataset-dir', str(DATASET_DIR),
    '--strict-root', str(STRICT_ROOT),
    '--full17-root', str(FULL17_ROOT),
    '--output-dir', str(OUTPUT_DIR),
]
preflight = subprocess.run(
    [sys.executable, str(REPO_DIR / 'scripts/evaluate_frozen_predictive_test.py'), *COMMON_ARGS, '--preflight-only'],
    cwd=REPO_DIR, check=True, capture_output=True, text=True,
)
print(preflight.stdout)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

decision = json.loads((OUTPUT_DIR / 'frozen_decision_snapshot.json').read_text())
inventory = pd.read_csv(OUTPUT_DIR / 'evaluated_checkpoint_inventory.csv')
assert decision['primary_candidate'] == 'strict_consensus'
assert decision['test_evaluation_candidates'] == ['strict_consensus', 'full17_reference']
assert len(inventory) == 20
assert set(inventory['checkpoint_name']) == {'best_model.pt'}
display(decision)
display(inventory[['candidate', 'model', 'seed', 'checkpoint_path', 'checkpoint_sha256']])
display(Markdown('**The held-out test split has not been opened by preflight.**'))

In [ ]:
CONFIRMATION = 'RUN_ONE_TIME_FROZEN_PREDICTIVE_TEST'
entered = input(f'Type {CONFIRMATION} to perform the one-time held-out inference: ').strip()
if entered != CONFIRMATION:
    raise RuntimeError('Confirmation mismatch. Held-out inference was not started.')

In [ ]:
evaluation = subprocess.run(
    [
        sys.executable, str(REPO_DIR / 'scripts/evaluate_frozen_predictive_test.py'),
        *COMMON_ARGS, '--device', 'auto', '--confirmation', entered,
    ],
    cwd=REPO_DIR, check=True, text=True,
)
print('One-time frozen evaluation command completed.')

In [ ]:
for name in [
    'test_candidate_aggregate.csv',
    'paired_test_seed_deltas.csv',
    'hierarchical_bootstrap_test_contrasts.csv',
]:
    display(Markdown(f'## {name}'))
    display(pd.read_csv(OUTPUT_DIR / name))
display(Markdown((OUTPUT_DIR / 'frozen_predictive_test_report.md').read_text()))
manifest = json.loads((OUTPUT_DIR / 'test_evaluation_manifest.json').read_text())
display(manifest)